# Advanced Tools

This notebook is a toolbox rather than a linear tutorial. After running Setup, each section lists its own prerequisites, and you can run only the tools relevant to your experiment. It covers:

1. **Create a New Persona**: generate, preview, and optionally filter synthetic SFT conversations for a custom character
2. **Tokenizer Fertility**: measure tokens per word across vocabulary sizes
3. **Select Pretraining Checkpoints**: evaluate, compare, and promote the best base model
4. **Select SFT Checkpoints**: evaluate, compare, and promote the best fine-tuned model
5. **Inference Internals**: chat completion API, context windows, and convenience wrappers
6. **Model Evaluation**: compare checkpoints with repeatable prompt sets and simple heuristics


## Prerequisites

> ⚠️ **Before running any tool:**
>
> 1. Complete the [Train from Scratch notebook](02_train_from_scratch.ipynb). It creates the datasets, tokenizer, and checkpoints expected here.
> 2. Run **Setup** whenever you start or restart the notebook kernel. It prepares the environment and moves the working directory to the repository root so commands such as `make` can find the correct `Makefile`.
>
> After completing the training prerequisite and running Setup, the remaining sections are independent and you can jump directly to the tools you need.


<details>
<summary>💡 Windows users</summary>

This notebook uses `make` to keep the advanced commands short. The recommended Windows path is WSL, where the cells work as written.

For native Windows, install GNU Make with a package manager such as Chocolatey or Scoop, then confirm that `make --version` works in the same terminal you use to start Jupyter. The underlying Python tools are cross-platform; only the `make` command shorthand requires this extra setup.

</details>

## Setup

**Google Colab (recommended):**

1. Open **Runtime > Change runtime type**.
2. Select **T4 GPU**, then click **Save**.
3. Run the Setup cell below, then continue to the tool you need.

**RunPod:** Start a GPU pod, then run the Setup cell below.

**Local laptop:**

1. If needed, [install uv](https://docs.astral.sh/uv/getting-started/installation/), then run `uv sync` from the repository root.
2. Select the project's `.venv` as your notebook kernel.
3. Run the Setup cell below.


In [ ]:
import os
import sys
from pathlib import Path

is_cloud = "google.colab" in sys.modules or "RUNPOD_POD_ID" in os.environ
if is_cloud:
    !pip install -q uv
    if not Path("Makefile").exists():
        !git clone --depth 1 https://github.com/khordoo/dusty-lm.git
        os.chdir("dusty-lm")
    !pip install -q -e .
elif not Path("Makefile").exists():
    # Walk up so make commands find the Makefile from any CWD
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "Makefile").exists():
            os.chdir(str(parent))
            break
    else:
        print("Run this from the repo root:\n        uv sync")

print(f"✅ Setup complete. Repo root: {Path.cwd()}")


## Inspect the Dataset Files

The training loop expects raw data under `artifacts/datasets/` and tokenized datasets beside it. This cell only reports what is present; it never downloads or replaces a dataset.

In [ ]:
from pathlib import Path

for path in [
    Path("artifacts/datasets/tinystories_base.txt"),
    Path("artifacts/datasets/dusty_sft.jsonl"),
]:
    if path.exists():
        print(f"\u2713 {path}: {path.stat().st_size / 1024 / 1024:.2f} MB")
    else:
        print(f"\u2717 {path}")

## 1. Create a New Persona


Pretraining teaches the base model how to use language. SFT gives it a persona: the character it plays, how it speaks, what it knows, and how it responds. To turn the base model into a different character, such as a toaster or a cat, you need conversations that demonstrate that behavior.

The scripts in this section use an external LLM to generate those synthetic conversations for you. After reviewing the new dataset, use it for a new SFT training run to produce the customized checkpoint.

To create a new persona, update the system prompt, `CATEGORIES`, `CATEGORY_DESCRIPTIONS`, and `PREFERRED_EXAMPLES` in [`data_pipeline/generate_sft.py`](../data_pipeline/generate_sft.py). These should all describe the character you want the model to imitate. Renaming Dusty without changing its SFT data will still teach the old robot-vacuum personality.

**Default provider:** OpenRouter (`https://openrouter.ai/api/v1`), which provides access to multiple models through one API. You can point `OPENAI_BASE_URL` at another OpenAI-compatible endpoint, such as OpenAI, Together AI, vLLM, Ollama, or LM Studio.

**Current generator models:**
- Primary: `qwen/qwen3-235b-a22b-2507:floor`
- Fallback: `openai/gpt-oss-120b:floor`

The Makefile sets these through `DUSTY_MODEL` and `DUSTY_FALLBACK_MODEL`. You can override either one in the `make` command. Use a capable instruction model and inspect a small sample before generating the full dataset.

**Before you start:**
- Set `OPENROUTER_API_KEY` or `OPENAI_API_KEY`. The next cell securely prompts for an OpenRouter key when neither variable is available.
- Set `OPENAI_BASE_URL` to override the default OpenRouter endpoint (e.g. `https://api.openai.com/v1`)
- Review the generated rows to confirm that the persona is consistent before scaling up

**Cost:** Our full Dusty dataset run used `qwen/qwen3-235b-a22b-2507:floor` and cost less than $3 at the time. Prices and provider routing can change. Start with 5 examples for each of the 65 categories, as below, to check quality before scaling up. The sample is written to separate files and never changes `dusty_sft.jsonl`, which is the canonical training dataset.

### 1.1 Generate SFT Data

Run the cell below to generate a small, isolated sample for your persona.

In [ ]:
import os
from getpass import getpass

if not (os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
os.environ.setdefault("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")

!make synthesize-sft \
  DUSTY_SFT_PER_CATEGORY=5 \
  DUSTY_SFT_BATCH_SIZE=5 \
  DUSTY_SFT_OUT=artifacts/datasets/dusty_sft_sample.jsonl \
  DUSTY_SFT_REJECTED=artifacts/datasets/dusty_sft_sample_rejected.jsonl

### 1.2 Review Accepted and Rejected Rows


Each accepted JSONL row needs `category`, `user`, and `dusty`. The `category` field is metadata for filtering and balancing; training uses the user and assistant text.

Rejected samples are candidates returned by the generator model that fail validation, so they are never added to the training data. Common reasons include missing fields, the wrong category, duplicate conversations, overly long responses, too many sentences, or words that do not fit the persona. The rejected file keeps the original candidate, its cleaned form when available, and an `errors` list.

A few rejected rows are normal, but repeated errors can reveal a pattern in the generator model's capabilities. For example, it may keep ignoring a category, writing responses that are too complex, or drifting out of character. Review several rejected rows together. If the same problem repeats, make the prompt and examples more explicit. If it continues after prompt tuning, try a stronger instruction model.

Generation is resumable, so rerunning the previous cell fills missing categories in the same sample instead of paying to recreate completed rows.

In [ ]:
import json
from pathlib import Path

sft_path = Path("artifacts/datasets/dusty_sft_sample.jsonl")
print("Accepted samples:\n")
with sft_path.open("r", encoding="utf-8") as file:
    for index, line in zip(range(5), file):
        row = json.loads(line)
        print(json.dumps(row, indent=2)[:800])
        print()

rejected_path = Path("artifacts/datasets/dusty_sft_sample_rejected.jsonl")
if rejected_path.exists():
    print("Rejected samples and reasons:\n")
    with rejected_path.open("r", encoding="utf-8") as file:
        for index, line in zip(range(3), file):
            row = json.loads(line)
            print(json.dumps(row, indent=2)[:1200])
            print()
else:
    print("No rejected samples were recorded in this run.")

### 1.3 Filter and Balance SFT Data [Optional]


Dusty uses a 256-token context window and the `<|im_end|>` marker to end each chat turn. Long assistant answers can crowd that marker out of a truncated training sequence and make stopping less reliable. The main safeguard is the prompt used to generate the data, which asks for concise responses.

The filter below is an optional inspection experiment. It measures assistant-answer length, removes answers above the selected threshold, and creates a category-balanced subset of the isolated sample. It does not replace `dusty_sft.jsonl` or change the data used by the `sft_dusty8m` profile.

To train on the filtered sample, open [`dustylm/config.py`](../dustylm/config.py), locate the `sft_dusty8m` profile, and change its `raw_sft_path` to `REPO_ROOT / "artifacts" / "datasets" / "dusty_sft_sample_filtered.jsonl"`. Then rerun SFT data preparation and training. Use the path to your full custom dataset instead when you scale beyond this small experiment. Keeping the custom data in a separate file is safer than renaming it or overwriting `dusty_sft.jsonl`.

> ⚠️ **Prerequisite:** This analysis needs the tokenizer created in the [training notebook](02_train_from_scratch.ipynb). The next cell checks for it but never retrains or overwrites it.

In [ ]:
from pathlib import Path

tokenizer_path = Path("artifacts/tokenizers/dusty_tokenizer.json")
if not tokenizer_path.exists():
    raise FileNotFoundError(
        "Tokenizer not found. Complete Notebook 02 before running this section."
    )
print(f"Using existing tokenizer: {tokenizer_path}")

In [ ]:
!make filter-sft \
  DUSTY_SFT_OUT=artifacts/datasets/dusty_sft_sample.jsonl \
  DUSTY_SFT_FILTER_TARGET=200 \
  DUSTY_SFT_MAX_ANSWER_TOKENS=48 \
  DUSTY_SFT_FILTERED_OUT=artifacts/datasets/dusty_sft_sample_filtered.jsonl

## 2. Tokenizer Fertility


> ⚠️ **Prerequisite:** This experiment reads `artifacts/datasets/tinystories_base.txt`, which is created in the [training notebook](02_train_from_scratch.ipynb).

Dusty has only about 8M parameters, so vocabulary size is an important design choice. A smaller vocabulary keeps the input embedding and output projection compact, leaving more of the parameter budget for the transformer layers, but it splits words into more tokens. A larger vocabulary represents text with fewer tokens, but it spends more parameters on storing vocabulary. This experiment helps find a practical balance between those two costs.

Fertility is the average number of tokens produced for each whitespace-delimited word. Higher fertility means the tokenizer splits words into more pieces and uses more of the context window. Lower fertility means fewer splits, but achieving it may require a larger vocabulary. For a tiny model, roughly `1.2` to `1.5` is usually a useful range.

The results below use the default 100K TinyStories tutorial slice. Each temporary tokenizer is trained on the complete slice, then measured on the first 10,000 non-empty lines. This matches the data used in the training notebook and makes the experiment reproducible for learners. A larger corpus may shift the exact values, but the vocabulary-size tradeoff remains the same.

| Vocab Size | Lines | Words | Tokens | Fertility |
|-----------|------:|------:|------:|---------:|
| 2,048 | 10,000 | 327,204 | 471,666 | 1.442 |
| 4,096 | 10,000 | 327,204 | 434,423 | 1.328 |
| 8,192 | 10,000 | 327,204 | 413,241 | 1.263 |

The table confirms the diminishing returns:

- **4,096 tokens:** 1.33 fertility at a small parameter footprint
- **8,192 tokens:** 1.26 fertility, but adds 2,097,152 parameters across the separate input embedding and output projection

For Dusty, moving from 4K to 8K did not produce a large enough improvement to justify the extra parameters, so the default tokenizer stays around 4K tokens.

In [ ]:
!uv run python data_pipeline/test_tokenizer_fertility.py \
  --input artifacts/datasets/tinystories_base.txt \
  --training-files artifacts/datasets/tinystories_base.txt \
  --lines 10000 \
  --vocab-sizes 2048 4096 8192

## 3. Select Pretraining Checkpoints

### 3.1 Evaluate Pretraining Checkpoints

The selected pretraining checkpoint becomes the language foundation for persona training. If the base model cannot produce coherent text, SFT cannot reliably turn it into a consistent character.

Evaluate the `dusty8m` pretraining checkpoints with story prompts and choose the one with the clearest language, strongest spelling, and least repetition. At this stage, test general text quality rather than Dusty's conversational personality.

In [ ]:
!make generate PROFILE=dusty8m PROMPT="Deep in the forest there was a " TOP_P=0.9 TEMPERATURE=0.8

### 3.2 Training Scale: Compute-Optimal vs. Inference-Aware

The original [Chinchilla scaling law](https://arxiv.org/abs/2203.15556) suggests roughly 20 training tokens per parameter as a compute-optimal reference. For an 8M parameter model, that is about 160M tokens. This is not a performance ceiling. When inference cost matters, training a smaller model for longer can be a better system-level tradeoff, an approach studied in [inference-aware scaling](https://proceedings.mlr.press/v235/sardana24a.html).

Modern models frequently train beyond the classic reference. For example, Meta trained [Llama 3 8B](https://github.com/meta-llama/llama3/blob/main/MODEL_CARD.md) on more than 15T tokens, roughly 1,875 tokens per parameter.

**Tutorial path:** The training notebook completes one epoch on a 100K TinyStories slice at about step 382 and promotes step 300, after roughly 17M tokens. This keeps the complete workflow practical on free Colab hardware while still producing a working base model that generates simple, coherent text.

**Published checkpoint:** DustyLM is designed to run locally in a browser, so maintaining a small inference footprint is central to the project. The official base checkpoint was trained on the full TinyStories split and selected at step 8,400. With batch size 224 and 256-token sequences, that represents approximately 482M token positions, or about three times the classic Chinchilla reference. This shifts more compute into one-time training while preserving fast inference with an 8M parameter model.

Repeating the 100K tutorial slice is not equivalent to scaling with more diverse data and can increase memorization. If you want to train longer, increase the amount and variety of your pretraining data instead of targeting a token count through repetition.

### 3.3 Compare Pretraining Checkpoints


During pretraining, the training loss curve helps narrow the search but does not tell the whole story. Do not choose checkpoints at random. Use TensorBoard to find the stable later region where the steep early drop has slowed and the curve begins to flatten, then compare several neighboring checkpoints from that region.

A lower training loss does not always mean better English. The checkpoint with the lowest loss might generate looping, repetitive text, while a slightly earlier checkpoint produces more natural phrasing.

To find the best version of your base model, evaluate the generated text manually. Use `CHECKPOINT_STEP` to sample outputs from different intermediate checkpoints. Generate a short story from several steps and compare them for coherence, spelling, and repetition.

In [ ]:
# Use unfinished story prompts to test the raw pretrained base model
base_prompts = [
    "Once upon a time,",
    "Lily was a little girl who loved to",
    "Deep in the forest, there lived a little boy named Timmy. One day, he",
]
base_checkpoint_steps = [200, 250, 300]

for step in base_checkpoint_steps:
    print("#" * 80)
    print("BASE CHECKPOINT_STEP:", step)
    for prompt in base_prompts:
        print("=" * 80)
        print("PROMPT:", prompt)
        !make generate PROFILE=dusty8m CHECKPOINT_STEP={step} PROMPT="{prompt}" TOP_P=0.9 TEMPERATURE=0.8

Sampling these checkpoints serves two different purposes depending on where you are in the training cycle:

- **Early Debugging (The Sanity Check)**

    While the model is actively training, sample early steps to verify that it is progressively learning. Early checkpoints will show the model transition from pure noise to forming real letters to generating basic words. You are simply confirming that the training pipeline is working.

- **Finding the Golden Zone (Post-Training)**

    Once training is complete, compare the checkpoints that were actually created by your run. With the current setup of 100K TinyStories, a batch size of 224, and one epoch, this means checking the later checkpoints around steps 200 to 300.

Sample several checkpoints strictly within your chosen region and compare them. The best checkpoint is the one that produces the most natural-sounding text with the fewest spelling errors and the least amount of looping.

<details>
<summary>💡 How larger training runs evaluate checkpoints</summary>

For larger training runs, teams do not just eyeball outputs. They hold out a validation dataset to measure perplexity without updating weights, and they rely on automated benchmark suites to rigorously evaluate capabilities.

We intentionally avoid a heavy validation pipeline here to keep this educational workflow fast and dependency-free. For a tiny 8M parameter model, using fixed prompts for a qualitative "vibes check" is a perfectly valid and highly efficient way to spot looping or gibberish without bloating the codebase.

</details>


### 3.4 Promote the Best Pretraining Checkpoint


Once you have evaluated your checkpoints and selected the winner from your pretraining run, you must **promote** it to be the official base model.

The subsequent Supervised Fine-Tuning (SFT) phase is hardcoded to look for a specific base model file named `artifacts/checkpoints/dusty8m.pt`. To promote your best checkpoint, copy your chosen step file and rename it to this target filename.

Run the following command to promote your chosen step (replace `300` if you selected a different one):

In [ ]:
from pathlib import Path
from shutil import copy2

BEST_BASE_STEP = 300
source = Path(f"artifacts/checkpoints/dusty8m_step_{BEST_BASE_STEP}.pt")
destination = Path("artifacts/checkpoints/dusty8m.pt")
if not source.exists():
    raise FileNotFoundError(f"Checkpoint not found: {source}")
copy2(source, destination)
print(f"Promoted {source} to {destination}")

**Sanity Check Your Base Model**

After promoting the file, it is highly recommended to run one final sanity check. This ensures that you copied the correct checkpoint and that `dusty8m.pt` is successfully generating coherent English before you invest time into the fine-tuning phase.

Run the generation command using the base profile to test your newly promoted model:

In [ ]:
!make generate PROFILE=dusty8m PROMPT="Deep in the forest there was a" TOP_P=0.9 TEMPERATURE=0.8

## 4. Select SFT Checkpoints

### 4.1 Compare SFT Step Checkpoints


Use the same selection approach as pretraining: inspect TensorBoard for the later region where the loss curve begins to flatten, then compare neighboring checkpoints from that region. During SFT, evaluate those checkpoints for **instruction following, persona consistency, repetition, and appropriate answer length**.

Unlike pretraining, SFT teaches the base model a conversational format and persona. Small models can overfit to chat templates quickly, so qualitative testing matters. With a batch size of 224 and the two-epoch tutorial setup, training ends near step 300. Compare step 200 with nearby checkpoints from the same run.

If an intermediate checkpoint behaves better than the final one, promote it over the profile's default checkpoint path. Normal `make generate PROFILE=sft_dusty8m ...` then uses the selected model without `CHECKPOINT_STEP`.

In [ ]:
prompts = [
    "where are you?",
    "are you scared?",
    "what do you dream about?",
]

checkpoint_steps = [150, 200, 250]  # Update these to match checkpoints from your run

for step in checkpoint_steps:
    print("#" * 80)
    print("CHECKPOINT_STEP:", step)
    for prompt in prompts:
        print("=" * 80)
        print("PROMPT:", prompt)
        !make generate PROFILE=sft_dusty8m CHECKPOINT_STEP={step} PROMPT="{prompt}" TOP_P=0.9 TEMPERATURE=0.8

### 4.2 Promote the Selected SFT Checkpoint

Exactly as you did with the base model, you must promote your winning checkpoint to finalize the pipeline. This copies your chosen step file and renames it to `artifacts/checkpoints/dusty8m_sft.pt`, which is the default weight for the `sft_dusty8m` profile used in the chat interface.

Set `BEST_STEP` to the checkpoint step that produced the best qualitative behavior. Run the following commands to promote your model and perform a final sanity check (replace `200` if you chose a different step):

In [ ]:
from pathlib import Path
from shutil import copy2

BEST_STEP = 200
source = Path(f"artifacts/checkpoints/dusty8m_sft_step_{BEST_STEP}.pt")
destination = Path("artifacts/checkpoints/dusty8m_sft.pt")
if not source.exists():
    raise FileNotFoundError(f"Checkpoint not found: {source}")
copy2(source, destination)
print(f"Promoted {source} to {destination}")

# This command intentionally omits CHECKPOINT_STEP. It now uses the promoted default checkpoint.
!make generate PROFILE=sft_dusty8m PROMPT="where are you?" TOP_P=0.9 TEMPERATURE=0.8

## 5. Inference Internals

### 5.1 Load an Inference Engine

Use the local promoted SFT checkpoint if you trained one. If you skipped training, pretrained checkpoints are available on Hugging Face Hub at `mkhordoo/dusty-8m-sft` and will be downloaded automatically when you run the cell below.

In [ ]:
from pathlib import Path
from dustylm.generate import get_device
from dustylm.inference import Inference

checkpoint_path = Path("artifacts/checkpoints/dusty8m_sft.pt")
tokenizer_path = Path("artifacts/tokenizers/dusty_tokenizer.json")

if not checkpoint_path.exists() or not tokenizer_path.exists():
    from huggingface_hub import hf_hub_download

if not checkpoint_path.exists():
    print("Local checkpoint not found, downloading from Hugging Face Hub...")
    checkpoint_path = Path(
        hf_hub_download(repo_id="mkhordoo/dusty-8m-sft", filename="model.pt")
    )

if not tokenizer_path.exists():
    print("Local tokenizer not found, downloading from Hugging Face Hub...")
    tokenizer_path = Path(
        hf_hub_download(
            repo_id="mkhordoo/dusty-8m-sft", filename="tokenizer.json"
        )
    )

engine = Inference(
    checkpoint_path=checkpoint_path,
    tokenizer_path=tokenizer_path,
    device=get_device(),
)
print("Profile:", engine.profile_name)

### 5.2 Convenience Wrapper and CLI Chat


For scripts or demos, wrap the response dictionary and return only the generated assistant text. For a terminal chat loop, use `make chat`, which runs `uv run python -m dustylm.inference`.

In [ ]:
def chat(prompt, *, max_tokens=128, temperature=0.8, top_p=0.8):
    response = engine.chat_completion(
        [{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in [
    "who are you?",
    "are you scared?",
    "what do you dream about?",
]:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")

### 5.3 OpenAI-Style Chat Completion


`Inference.chat_completion()` accepts OpenAI-style `messages` and returns an OpenAI-like dictionary. That keeps local DustyLM inference close to common chat-completion calling patterns while still running against a local PyTorch checkpoint.

In [ ]:
messages = [{"role": "user", "content": "where are you?"}]

response = engine.chat_completion(
    messages=messages,
    max_tokens=64,
    temperature=0.8,
    top_p=0.8,
)

response

In [ ]:
assistant_text = response["choices"][0]["message"]["content"]
print("Dusty>", assistant_text)

### 5.4 Short Context Windows for Tiny Models


Dusty is only about 8M parameters, so the default chat history window is intentionally tiny: `max_chat_turns=1`. That keeps the current user request from being diluted by old conversation tokens. Larger SFT profiles, such as SmolLM2-based models, can use a larger window. The Python example below compares the default with a controlled two-turn override.

In [ ]:
print("Profile:", engine.profile_name)
print("Default max_chat_turns:", engine.spec.max_chat_turns)

history = [
    {"role": "system", "content": "Answer as Dusty, a tiny vacuum robot."},
    {"role": "user", "content": "what is under the couch?"},
    {"role": "assistant", "content": "crumbs. many crumbs."},
    {"role": "user", "content": "should you go there?"},
]

response = engine.chat_completion(history, max_tokens=128)
print("Default window:", response["choices"][0]["message"]["content"])

response = engine.chat_completion(history, max_tokens=128, max_chat_turns=2)
print("Two-turn window:", response["choices"][0]["message"]["content"])

You can also use the same two-turn window in the interactive terminal:

```bash
make chat MAX_CHAT_TURNS=2
```

This lets each response use the current user prompt and one previous user-assistant exchange. We do not recommend setting this above `2` for Dusty. Its 256-token context window leaves little room for longer histories, which can reduce response quality.

### 5.5 The DustyLM SDK

The previous sections used this repository's `Inference` class to load raw `.pt` weights and expose the generation engine. For applications, the separate `dustylm-sdk` package provides a smaller public interface for the same workflow.

Run the installation cell once, then load either the checkpoint produced by this repository or the published Hugging Face model.

In [ ]:
!uv pip install --python "{sys.executable}" "dustylm-sdk>=0.1.2"

**Local checkpoint produced by this repository:**

In [ ]:
from dustylm_sdk import DustyLM

local_model = DustyLM.from_pretrained(
    "artifacts/checkpoints",
    model_file="dusty8m_sft.pt",
    tokenizer_file="../tokenizers/dusty_tokenizer.json",
)
response = local_model.chat(
    [{"role": "user", "content": "who are you?"}]
)
print(response["choices"][0]["message"]["content"])

**Published model from Hugging Face Hub:**

In [ ]:
from dustylm_sdk import DustyLM

published_model = DustyLM.from_pretrained("mkhordoo/dusty-8m-sft")
response = published_model.chat(
    [{"role": "user", "content": "who are you?"}]
)
print(response["choices"][0]["message"]["content"])

For compatible DustyLM PyTorch checkpoints, the SDK infers several architecture dimensions from state-dict shapes. It supports both PyTorch and ONNX backends. See the [DustyLM SDK documentation](https://github.com/khordoo/dustylm-sdk) for backend options and more examples.

## 6. Model Evaluation

Instead of judging a checkpoint from one response, use fixed prompts and repeated generations to compare behavior across training steps.

### 6.1 Automated Checkpoint Comparison

Run the same small prompt set against several checkpoints and inspect the results side by side. The evaluator saves both JSON and CSV reports under `artifacts/evaluations/checkpoints/`.

> ⚠️ **Prerequisite:** The checkpoint steps and tokenizer must come from the same training run. The tutorial uses SFT steps `150`, `200`, and `250`. Update the steps below if your run created different checkpoints.

The first cell creates a six-prompt smoke-test set. Replace these prompts when evaluating a custom persona.

In [ ]:
import json
from pathlib import Path

quick_inputs = [
    {"id": 1, "category": "identity", "input": "who are you?"},
    {"id": 2, "category": "crumbs_food", "input": "did you find any crumbs today?"},
    {"id": 3, "category": "obstacles_dangers", "input": "what happens if you hit a sock?"},
    {"id": 4, "category": "battery_dock", "input": "where do you go when your battery is low?"},
    {"id": 5, "category": "personality_feelings", "input": "what do you dream about?"},
    {"id": 6, "category": "maker_identity", "input": "who made you?"},
]

quick_inputs_path = Path("artifacts/evaluations/notebook_sft_inputs.json")
quick_inputs_path.parent.mkdir(parents=True, exist_ok=True)
quick_inputs_path.write_text(json.dumps(quick_inputs, indent=2) + "\n")
quick_inputs_path

In [ ]:
!make eval-checkpoints \
  EVAL_PROFILE=sft_dusty8m \
  EVAL_STEPS="150 200 250" \
  EVAL_INPUTS=artifacts/evaluations/notebook_sft_inputs.json \
  EVAL_RUN_ID=notebook_sft \
  EVAL_TEMPERATURE=0.7 \
  EVAL_TOP_P=0.8 \
  EVAL_MAX_NEW_TOKENS=64

In [ ]:
import csv

report_path = Path("artifacts/evaluations/checkpoints/notebook_sft.csv")
with report_path.open(encoding="utf-8") as file:
    comparison_rows = list(csv.DictReader(file))

preview_prompts = ["who are you?", "what do you dream about?", "who made you?"]
steps = [150, 200, 250]
outputs = {
    (row["input"], int(row["checkpoint_step"])): row["output"]
    for row in comparison_rows
}

def table_text(value):
    return value.replace("|", "\\|").replace("\n", " ")

table = [
    "| Prompt | Step 150 | Step 200 | Step 250 |",
    "|---|---|---|---|",
]
for prompt in preview_prompts:
    cells = [table_text(outputs[(prompt, step)]) for step in steps]
    table.append(f"| {prompt} | {cells[0]} | {cells[1]} | {cells[2]} |")

def display_markdown(lines):
    text = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
    except ImportError:
        print(text)
    else:
        display(Markdown(text))

display_markdown(table)

The verified runs show why several samples matter. Step `150` often missed maker identity. Step `200` had stable Dusty identity but sometimes missed the maker question. Step `250` improved maker identity but occasionally confused the maker with Dusty's identity or repeated phrases. Sampling can vary, so compare the tradeoffs across multiple prompts before promoting a checkpoint.

### 6.2 Structured Consistency Check

A single response can be lucky. The consistency tool generates each focused prompt three times per checkpoint so you can inspect variation, contradictions, and repeated phrasing.

In [ ]:
!uv run python evaluation/check_consistency.py \
  --profile sft_dusty8m \
  --steps 150 200 250 \
  --runs 3 \
  --temperature 0.7 \
  --top-p 0.8 \
  --output artifacts/evaluations/notebook_consistency.csv

In [ ]:
from collections import defaultdict

consistency_path = Path("artifacts/evaluations/notebook_consistency.csv")
with consistency_path.open(encoding="utf-8") as file:
    consistency_rows = list(csv.DictReader(file))

responses = defaultdict(list)
for row in consistency_rows:
    key = (int(row["checkpoint_step"]), row["prompt"])
    responses[key].append(row["output"].strip())

summary = [
    "| Checkpoint | Average distinct responses out of 3 | Prompts identical across all runs |",
    "|---:|---:|---:|",
]
for step in [150, 200, 250]:
    groups = [values for (group_step, _), values in responses.items() if group_step == step]
    average_unique = sum(len(set(values)) for values in groups) / len(groups)
    identical = sum(len(set(values)) == 1 for values in groups)
    summary.append(f"| {step} | {average_unique:.2f} | {identical} |")

display_markdown(summary)

Lower variation is not automatically better. Identical answers may indicate stable behavior, but they can also reveal memorization or repetitive generation. Always inspect the underlying responses alongside the summary.

The `evaluation/` directory also includes the full SFT and base prompt sets, web-demo topic checks, emotion-keyword counts, and possible contradiction cases for manual review. These are useful heuristics, not comprehensive model-quality scores.

**[Read the Evaluation README](../evaluation/README.md) for the complete CLI reference and methodology.**